In [0]:
spark.version


'4.2.0'

In [0]:
%sql
-- Завдання 2.1. Поштові індекси посадки

SELECT 
    pickup_zip,
    COUNT(*) AS trips_count,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
HAVING trips_count >= 100
ORDER BY trips_count DESC



pickup_zip,trips_count,avg_distance,avg_fare
10001,1227,2.21,10.62
10003,1181,2.32,10.98
10011,1129,2.29,10.91
10021,1021,2.02,10.21
10018,1012,2.6,11.4
10023,1008,2.12,10.04
10028,929,2.2,10.21
10012,834,2.43,11.35
10110,763,2.31,10.9
10065,702,1.96,9.81


In [0]:
%sql
-- Завдання 2.2. Денна виручка з накопиченням

WITH daily_revenue AS (
SELECT 
    CAST(tpep_pickup_datetime AS DATE) AS pickup_date,
    COUNT(tpep_pickup_datetime) AS trips_count,
    SUM(fare_amount) AS total_fare
FROM samples.nyctaxi.trips
GROUP BY CAST(tpep_pickup_datetime AS DATE)
)

SELECT
    pickup_date,
    trips_count,
    total_fare,
    ROUND(SUM(total_fare) OVER (PARTITION BY YEAR(pickup_date), MONTH(pickup_date)
        ORDER BY pickup_date), 2) AS cumulative_fare
FROM daily_revenue
ORDER BY pickup_date

pickup_date,trips_count,total_fare,cumulative_fare
2016-01-01,326,3998.0,3998.0
2016-01-02,338,4074.0,8072.0
2016-01-03,276,3405.5,11477.5
2016-01-04,325,4011.3,15488.8
2016-01-05,359,4052.5,19541.3
2016-01-06,360,4187.5,23728.8
2016-01-07,356,4348.5,28077.3
2016-01-08,390,4934.0,33011.3
2016-01-09,412,4912.5,37923.8
2016-01-10,347,4154.5,42078.3


In [0]:
%sql
-- Завдання 2.3. Країни-лідери в регіонах 

WITH regions AS (
SELECT 
    r.r_name AS region,
    n.n_name AS nation,
    SUM(o.o_totalprice) AS total_revenue
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation n ON n.n_nationkey = c.c_nationkey
JOIN samples.tpch.region r ON r.r_regionkey = n.n_regionkey
WHERE YEAR(o.o_orderdate) = 1995
GROUP BY n.n_name, r.r_name
),
ranked AS (
SELECT
    region,
    nation,
    total_revenue,
    RANK() OVER(PARTITION BY region ORDER BY total_revenue DESC) AS revenue_rank
FROM regions
)

SELECT 
    region,
    nation,
    total_revenue,
    revenue_rank
FROM ranked
WHERE revenue_rank <= 3
ORDER BY region, revenue_rank 

region,nation,total_revenue,revenue_rank
AFRICA,ETHIOPIA,6870438595.05,1
AFRICA,KENYA,6829830997.18,2
AFRICA,MOZAMBIQUE,6825859913.59,3
AMERICA,ARGENTINA,6924104098.33,1
AMERICA,CANADA,6819698616.46,2
AMERICA,UNITED STATES,6813047408.62,3
ASIA,INDIA,6916650512.50,1
ASIA,INDONESIA,6913135399.36,2
ASIA,CHINA,6876145532.41,3
EUROPE,FRANCE,6963662248.61,1
